In [0]:
SILVER_TABLE = "workspace.default.silver_rides"

# Load silver table and register as SQL view
df = spark.table(SILVER_TABLE)
df.createOrReplaceTempView("rides")

print(f"Loaded {df.count():,} rows from Silver table")
print("Ready to query as 'rides'")

In [0]:
q1 = spark.sql("""
    SELECT 
        member_casual,
        COUNT(ride_id)                    AS total_rides,
        ROUND(AVG(ride_time_min), 2)      AS avg_duration_min,
        ROUND(AVG(ride_time_min) / 60, 2) AS avg_duration_hrs
    FROM rides
    GROUP BY member_casual
    ORDER BY member_casual
""")

print("Q1: Total rides and average trip duration by rider type")
display(q1)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
q2 = spark.sql("""
    SELECT 
        member_casual,
        rideable_type,
        COUNT(ride_id)               AS total_rides,
        ROUND(AVG(ride_time_min), 2) AS avg_duration_min
    FROM rides
    GROUP BY member_casual, rideable_type
    ORDER BY member_casual, rideable_type
""")
print("Q2: Rides by ride type")
display(q2)
q2.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q2_ridetype")

In [0]:
q3 = spark.sql("""
    SELECT 
        member_casual,
        month_num,
        month_name,
        COUNT(ride_id) AS total_rides
    FROM rides
    GROUP BY member_casual, month_num, month_name
    ORDER BY member_casual, month_num
""")
print("Q3: Rides per month")
display(q3)
q3.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q3_monthly")

Databricks visualization. Run in Databricks to view.

In [0]:
q4 = spark.sql("""
    SELECT 
        member_casual,
        weekday,
        COUNT(ride_id) AS total_rides
    FROM rides
    GROUP BY member_casual, weekday
    ORDER BY member_casual, weekday
""")
print("Q4: Rides per weekday")
display(q4)
q4.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q4_weekday")

Databricks visualization. Run in Databricks to view.

In [0]:
q5 = spark.sql("""
    SELECT 
        member_casual,
        hour_of_day,
        hour_label,
        COUNT(ride_id) AS total_rides
    FROM rides
    GROUP BY member_casual, hour_of_day, hour_label
    ORDER BY member_casual, hour_of_day
""")
print("Q5: Rides per hour")
display(q5)
q5.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q5_hourly")

Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 member starting stations
q7 = spark.sql("""
    SELECT start_station_name, COUNT(ride_id) AS rides
    FROM rides WHERE TRIM(LOWER(member_casual)) = 'member'
    GROUP BY start_station_name ORDER BY rides DESC LIMIT 10
""")
print("Q7: Top 10 member starting stations")
display(q7)
q7.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q7_member_start")

# Top 10 member ending stations
q8 = spark.sql("""
    SELECT end_station_name, COUNT(ride_id) AS rides
    FROM rides WHERE TRIM(LOWER(member_casual)) = 'member'
    GROUP BY end_station_name ORDER BY rides DESC LIMIT 10
""")
print("Q8: Top 10 member ending stations")
display(q8)
q8.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q8_member_end")

# Top 10 casual starting stations
q9 = spark.sql("""
    SELECT start_station_name, COUNT(ride_id) AS rides
    FROM rides WHERE TRIM(LOWER(member_casual)) = 'casual'
    GROUP BY start_station_name ORDER BY rides DESC LIMIT 10
""")
print("Q9: Top 10 casual starting stations")
display(q9)
q9.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q9_casual_start")

# Top 10 casual ending stations
q10 = spark.sql("""
    SELECT end_station_name, COUNT(ride_id) AS rides
    FROM rides WHERE TRIM(LOWER(member_casual)) = 'casual'
    GROUP BY end_station_name ORDER BY rides DESC LIMIT 10
""")
print("Q10: Top 10 casual ending stations")
display(q10)
q10.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q10_casual_end")

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Most popular member stations by hour
q13 = spark.sql("""
    WITH start_counts AS (
        SELECT start_station_name AS station, hour_label, hour_of_day,
               COUNT(ride_id) AS start_rides
        FROM rides WHERE TRIM(LOWER(member_casual)) = 'member'
        GROUP BY start_station_name, hour_label, hour_of_day
    ),
    end_counts AS (
        SELECT end_station_name AS station, hour_label, hour_of_day,
               COUNT(ride_id) AS end_rides
        FROM rides WHERE TRIM(LOWER(member_casual)) = 'member'
        GROUP BY end_station_name, hour_label, hour_of_day
    ),
    combined AS (
        SELECT COALESCE(s.station, e.station)       AS station,
               COALESCE(s.hour_label, e.hour_label) AS hour_label,
               COALESCE(s.hour_of_day, e.hour_of_day) AS hour_of_day,
               COALESCE(s.start_rides, 0) + COALESCE(e.end_rides, 0) AS total_trips
        FROM start_counts s
        FULL OUTER JOIN end_counts e
          ON s.station = e.station AND s.hour_of_day = e.hour_of_day
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY hour_of_day ORDER BY total_trips DESC) AS rn
        FROM combined
    )
    SELECT station, hour_label, hour_of_day, total_trips
    FROM ranked WHERE rn = 1
    ORDER BY hour_of_day
""")
print("Q13: Most popular member station per hour")
display(q13)
q13.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q13_member_hourly")

# Most popular casual stations by hour
q14 = spark.sql("""
    WITH start_counts AS (
        SELECT start_station_name AS station, hour_label, hour_of_day,
               COUNT(ride_id) AS start_rides
        FROM rides WHERE TRIM(LOWER(member_casual)) = 'casual'
        GROUP BY start_station_name, hour_label, hour_of_day
    ),
    end_counts AS (
        SELECT end_station_name AS station, hour_label, hour_of_day,
               COUNT(ride_id) AS end_rides
        FROM rides WHERE TRIM(LOWER(member_casual)) = 'casual'
        GROUP BY end_station_name, hour_label, hour_of_day
    ),
    combined AS (
        SELECT COALESCE(s.station, e.station)         AS station,
               COALESCE(s.hour_label, e.hour_label)   AS hour_label,
               COALESCE(s.hour_of_day, e.hour_of_day) AS hour_of_day,
               COALESCE(s.start_rides, 0) + COALESCE(e.end_rides, 0) AS total_trips
        FROM start_counts s
        FULL OUTER JOIN end_counts e
          ON s.station = e.station AND s.hour_of_day = e.hour_of_day
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY hour_of_day ORDER BY total_trips DESC) AS rn
        FROM combined
    )
    SELECT station, hour_label, hour_of_day, total_trips
    FROM ranked WHERE rn = 1
    ORDER BY hour_of_day
""")
print("Q14: Most popular casual station per hour")
display(q14)
q14.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q14_casual_hourly")

In [0]:
# Most popular casual routes
q15 = spark.sql("""
    SELECT start_station_name, end_station_name, COUNT(ride_id) AS rides
    FROM rides WHERE TRIM(LOWER(member_casual)) = 'casual'
    GROUP BY start_station_name, end_station_name
    ORDER BY rides DESC LIMIT 20
""")
print("Q15: Most popular casual routes")
display(q15)
q15.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q15_casual_routes")

# Most popular member routes
q16 = spark.sql("""
    SELECT start_station_name, end_station_name, COUNT(ride_id) AS rides
    FROM rides WHERE TRIM(LOWER(member_casual)) = 'member'
    GROUP BY start_station_name, end_station_name
    ORDER BY rides DESC LIMIT 20
""")
print("Q16: Most popular member routes")
display(q16)
q16.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q16_member_routes")

In [0]:
q17_start = spark.sql("""
    SELECT 
        start_station_name,
        member_casual,
        hour_of_day,
        ROUND(AVG(start_lat), 6) AS avg_lat,
        ROUND(AVG(start_lng), 6) AS avg_lng,
        COUNT(ride_id)           AS rides
    FROM rides
    GROUP BY start_station_name, member_casual, hour_of_day
    ORDER BY start_station_name, member_casual, hour_of_day
""")
print("Q17a: Station coordinates by rider type and hour")
display(q17_start.limit(10))
q17_start.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_q17_station_coords")# Holiday rides summary
q_holiday = spark.sql("""
    SELECT 
        CASE 
            WHEN DATE(started_at) = '2024-09-02' THEN 'Labor Day'
            WHEN DATE(started_at) = '2024-10-14' THEN 'Columbus Day'
            WHEN DATE(started_at) = '2024-11-11' THEN 'Veterans Day'
            WHEN DATE(started_at) = '2024-11-28' THEN 'Thanksgiving'
            WHEN DATE(started_at) = '2024-12-25' THEN 'Christmas'
            WHEN DATE(started_at) = '2025-01-01' THEN 'New Years Day'
            WHEN DATE(started_at) = '2025-01-20' THEN 'MLK Day'
            WHEN DATE(started_at) = '2025-02-17' THEN 'Presidents Day'
            WHEN DATE(started_at) = '2025-05-26' THEN 'Memorial Day'
            WHEN DATE(started_at) = '2025-06-19' THEN 'Juneteenth'
            WHEN DATE(started_at) = '2025-07-04' THEN 'Independence Day'
            WHEN DATE(started_at) = '2025-09-01' THEN 'Labor Day 2025'
        END                          AS holiday_name,
        DATE(started_at)             AS holiday_date,
        member_casual,
        COUNT(ride_id)               AS total_rides,
        ROUND(AVG(ride_time_min), 2) AS avg_duration_min,
        MIN(ride_time_min)           AS min_duration,
        MAX(ride_time_min)           AS max_duration
    FROM rides
    WHERE is_holiday = true
    GROUP BY DATE(started_at), member_casual
    ORDER BY holiday_date, member_casual
""")
print("Holiday rides summary")
display(q_holiday)
q_holiday.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_holiday_summary")

# Holiday vs average day comparison
q_holiday_vs_avg = spark.sql("""
    WITH holiday_stats AS (
        SELECT member_casual,
               ROUND(AVG(ride_time_min), 2) AS holiday_avg_duration,
               COUNT(ride_id)               AS holiday_rides
        FROM rides WHERE is_holiday = true
        GROUP BY member_casual
    ),
    normal_stats AS (
        SELECT member_casual,
               ROUND(AVG(ride_time_min), 2) AS normal_avg_duration,
               COUNT(ride_id)               AS normal_rides
        FROM rides WHERE is_holiday = false
        GROUP BY member_casual
    )
    SELECT 
        h.member_casual,
        h.holiday_rides,
        h.holiday_avg_duration,
        n.normal_rides,
        n.normal_avg_duration,
        ROUND(h.holiday_avg_duration - n.normal_avg_duration, 2) AS duration_difference
    FROM holiday_stats h
    JOIN normal_stats n ON h.member_casual = n.member_casual
""")
print("Holiday vs average day comparison")
display(q_holiday_vs_avg)
q_holiday_vs_avg.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_holiday_vs_avg")

Databricks visualization. Run in Databricks to view.

In [0]:
# List all gold tables we created
gold_tables = [
    "workspace.default.gold_q1_overview",
    "workspace.default.gold_q2_ridetype",
    "workspace.default.gold_q3_monthly",
    "workspace.default.gold_q4_weekday",
    "workspace.default.gold_q5_hourly",
    "workspace.default.gold_q7_member_start",
    "workspace.default.gold_q8_member_end",
    "workspace.default.gold_q9_casual_start",
    "workspace.default.gold_q10_casual_end",
    "workspace.default.gold_q13_member_hourly",
    "workspace.default.gold_q14_casual_hourly",
    "workspace.default.gold_q15_casual_routes",
    "workspace.default.gold_q16_member_routes",
    "workspace.default.gold_q17_station_coords",
    "workspace.default.gold_holiday_summary",
    "workspace.default.gold_holiday_vs_avg"
]

print(f"{'Table':<45} {'Rows':>8}")
print("-" * 55)
for table in gold_tables:
    try:
        count = spark.table(table).count()
        print(f"{table.split('.')[-1]:<45} {count:>8,}")
    except Exception as e:
        print(f"{table.split('.')[-1]:<45} ERROR")

print("-" * 55)
print(f"\nTotal gold tables: {len(gold_tables)}")

In [0]:
# Print key findings from the gold tables
q1 = spark.table("workspace.default.gold_q1_overview").toPandas()
q_hol = spark.table("workspace.default.gold_holiday_vs_avg").toPandas()

print("=" * 55)
print("CYCLISTIC KEY FINDINGS")
print("=" * 55)

for _, row in q1.iterrows():
    print(f"\n{row['member_casual'].upper()} RIDERS")
    print(f"  Total rides:       {row['total_rides']:>10,.0f}")
    print(f"  Avg duration:      {row['avg_duration_min']:>10.1f} min")

print("\n" + "=" * 55)
print("HOLIDAY vs NORMAL DAY")
print("=" * 55)
for _, row in q_hol.iterrows():
    print(f"\n{row['member_casual'].upper()}")
    print(f"  Holiday avg duration: {row['holiday_avg_duration']:>6.1f} min")
    print(f"  Normal avg duration:  {row['normal_avg_duration']:>6.1f} min")
    print(f"  Difference:           {row['duration_difference']:>+6.1f} min")